In [1]:
MODEL_CODE = "dt"
MODEL_NAME = "Decision Tree"

# 07: Decision Tree Modeli

**Proje:** Framingham Kalp Çalışması verisi ile inme (stroke) tahmini · **Model ailesi:** ağaç / bagging

**Decision Tree**, veriyi "AGE > 60 mı?", "SYSBP > 150 mi?" gibi evet/hayır sorularıyla tekrar tekrar ikiye böler ve her yaprakta bir karar verir. Kararı bir soru dizisi olduğu için doğrudan okunabilen bir modeldir. Sınır verilmezse her yaprakta tek bir sınıf kalana kadar böler ve eğitim verisini ezberler (overfitting); bu yüzden ağacın boyu hiperparametrelerle sınırlanır.

**Model neyi kendisi öğrenir, neyi biz seçeriz?** Hangi sütunda hangi eşikle bölüneceğini model eğitim sırasında kendisi bulur. Ağacın ne kadar derinleşeceği, bir yaprakta en az kaç kişi kalacağı ve dengesizliğin nasıl giderileceği ise eğitimden önce verilmesi gereken ayarlardır (hiperparametre). Bunlar GridSearch ile tüm kombinasyonlar denenerek, ortak CV katları üzerinde seçilir.

**Bu notebook'ta yapılanlar:** hiperparametre ve dengeleme araması · derinliğin overfitting'e etkisi · feature engineering'in etkisi · final modelin CV değerlendirmesi, eşik seçimi, öğrenme eğrisi · ağacın ilk seviyeleri · test seti · XAI (SHAP + LIME)

**Colab'da çalıştırmak için:** Sol menüdeki klasör simgesinden `train.csv`, `test.csv`, `config.py`, `preprocessing.py`, `metrics.py` ve `explain.py` dosyalarını yükleyin.

## 1. Kurulum

In [2]:
# shap and lime are not preinstalled on Colab
try:
    import lime
    import shap
except ImportError:
    %pip install -q shap lime

In [3]:
import sys
from pathlib import Path

# Repo: ../src. Colab: the src/*.py files uploaded to the working directory.
SRC_DIR = next((p for p in (Path("../src"), Path("src"), Path(".")) if (p / "config.py").exists()), None)
if SRC_DIR is None:
    raise FileNotFoundError("config.py bulunamadı. Colab'da src klasöründeki .py dosyalarını sol menüden yükleyin.")
sys.path.insert(0, str(SRC_DIR.resolve()))

# Output folders: models/ and figures/ at the repo root (the working directory on Colab)
ROOT = SRC_DIR.resolve().parent if SRC_DIR.resolve().name == "src" else Path(".")
MODEL_DIR = ROOT / "models";  MODEL_DIR.mkdir(exist_ok=True)
FIG_DIR = ROOT / "figures";   FIG_DIR.mkdir(exist_ok=True)

import time
import warnings

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree

from config import CLASS_NAMES, MAIN_METRIC, RANDOM_STATE
from explain import explain
from metrics import (SCORING, cv_evaluate, learning_curve_analysis, shared_cv, test_evaluate,
                     threshold_analysis)
from preprocessing import build_pipeline, load_data

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

## 2. Veri

`train.csv` / `test.csv`. Ön işleme (feature engineering, encoding, ölçekleme, SMOTE) `build_pipeline()` içinde yapılır; test seti en sona kadar kullanılmaz.

In [4]:
X_train, X_test, y_train, y_test = load_data()
print("Eğitim:", X_train.shape, f"| inme: {y_train.sum()} kişi ({y_train.mean():.2%})")
print("Test:  ", X_test.shape, f"| inme: {y_test.sum()} kişi ({y_test.mean():.2%})")

Eğitim: (3521, 17) | inme: 306 kişi (8.69%)
Test:   (881, 17) | inme: 77 kişi (8.74%)


## 3. Hiperparametre ve dengeleme araması

| Parametre | Değerler | Anlamı |
|---|---|---|
| `max_depth` | 2, 3, 4, 5, 6, 8, 10, None | Ağacın en fazla kaç seviye derinleşeceği. Derin ağaç eğitim verisine sıkı uyar (overfitting), `None` sınırsız demektir |
| `min_samples_leaf` | 1, 5, 10, 20, 40, 80 | Bir yaprakta en az kaç kişi kalacağı. Büyük değer, tek tük hastaya özel kural öğrenilmesini engeller |
| `criterion` | gini, entropy | Bölmenin iyiliğini ölçen formül; ikisi genelde yakın sonuç verir |
| dengeleme | SMOTE, class_weight | SMOTE azınlık sınıfı için sentetik hasta üretir. class_weight veri üretmez, inme vakalarındaki hatayı daha ağır cezalandırır. Ağaçlar SMOTE'un sentetik örneklerini ezberleyebildiği için ikisi de denenir |

Dengeleme de aramanın bir parçasıdır: SMOTE'lu satırlarda pipeline'ın `balance` adımı SMOTE, class_weight'li satırlarda bu adım boştur (`passthrough`) ve ağaca `class_weight="balanced"` verilir. Böylece 192 kombinasyonun tamamı aynı CV katları (`shared_cv()`) ve aynı metriklerle (`SCORING`) karşılaştırılır; en iyi model `MAIN_METRIC` (PR-AUC) ile seçilir.

In [5]:
def search_table(search, params):
    """Hiperparametre denemeleri: eğitim ve CV skoru, aradaki fark (overfitting göstergesi)."""
    r = pd.DataFrame(search.cv_results_)
    t = r[[f"param_{p}" for p in params]].map(str)  # None stays as "None"
    t.columns = [p.replace("model__", "") for p in params]
    for m in [MAIN_METRIC, "recall", "roc_auc"]:
        t[f"{m}_cv"] = r[f"mean_test_{m}"]
    t[f"{MAIN_METRIC}_train"] = r[f"mean_train_{MAIN_METRIC}"]
    t["fark (train - cv)"] = t[f"{MAIN_METRIC}_train"] - t[f"{MAIN_METRIC}_cv"]
    return t.sort_values(f"{MAIN_METRIC}_cv", ascending=False).round(4)


def balancing_name(step):
    return "smote" if isinstance(step, SMOTE) else "class_weight"

In [6]:
tree_params = {
    "model__max_depth": [2, 3, 4, 5, 6, 8, 10, None],
    "model__min_samples_leaf": [1, 5, 10, 20, 40, 80],
    "model__criterion": ["gini", "entropy"],
}
params = [
    {**tree_params, "balance": [SMOTE(random_state=RANDOM_STATE)], "model__class_weight": [None]},
    {**tree_params, "balance": ["passthrough"], "model__class_weight": ["balanced"]},
]

start = time.time()
search = GridSearchCV(build_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE), balancing="smote"),
                      params, cv=shared_cv(), scoring=SCORING, refit=MAIN_METRIC,
                      n_jobs=-1, return_train_score=True)
search.fit(X_train, y_train)
best = search.best_estimator_   # the pipeline itself, not the search object
chosen_balancing = balancing_name(search.best_params_["balance"])
best_params = {k.replace("model__", ""): v for k, v in search.best_params_.items() if k.startswith("model__")}

print(f"Süre: {time.time() - start:.0f} sn")
print("Seçilen dengeleme:", chosen_balancing)
print("En iyi parametreler:", best_params)
print(f"En iyi CV {MAIN_METRIC}: {search.best_score_:.4f}")

results = search_table(search, ["balance", "model__max_depth", "model__min_samples_leaf", "model__criterion"])
results["balance"] = results["balance"].map(lambda s: "smote" if "SMOTE" in s else "class_weight")
results.head(15)

Süre: 11 sn
Seçilen dengeleme: smote
En iyi parametreler: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 5, 'min_samples_leaf': 80}
En iyi CV pr_auc: 0.1680


,balance,max_depth,min_samples_leaf,criterion,pr_auc_cv,recall_cv,roc_auc_cv,pr_auc_train,fark (train - cv)
71,smote,5,80,entropy,0.1680,0.5456,0.6622,0.1896,0.0216
118,class_weight,5,40,gini,0.1649,0.5948,0.6521,0.2392,0.0743
112,class_weight,4,40,gini,0.1645,0.6275,0.6659,0.2143,0.0498
153,class_weight,3,20,entropy,0.1640,0.7319,0.6762,0.1868,0.0228
106,class_weight,3,40,gini,0.1639,0.7418,0.6813,0.1888,0.0249
107,class_weight,3,80,gini,0.1636,0.6963,0.6802,0.1888,0.0252
161,class_weight,4,80,entropy,0.1634,0.6438,0.6610,0.2062,0.0428
154,class_weight,3,40,entropy,0.1632,0.7287,0.6760,0.1877,0.0245
105,class_weight,3,20,gini,0.1631,0.7417,0.6808,0.1843,0.0212
137,class_weight,10,80,gini,0.1631,0.6211,0.6522,0.2379,0.0748


### Derinliğin etkisi (overfitting)

Her derinlik için, iki dengeleme yönteminde de en iyi `min_samples_leaf` / `criterion` seçilerek eğitim ve CV skorları çizilir. Derinlik arttıkça eğitim skoru yükselir. CV skoru bir noktadan sonra yükselmiyor ya da düşüyorsa ağaç ezberlemeye başlamıştır: en iyi derinlik, iki çizginin ayrılmaya başladığı yerin civarıdır.

In [7]:
r = results.copy()
r["max_depth"] = r["max_depth"].replace("None", "sınırsız")
order = ["2", "3", "4", "5", "6", "8", "10", "sınırsız"]

fig, ax = plt.subplots(figsize=(8, 4.5))
for bal, color in [("smote", "#4C72B0"), ("class_weight", "#DD8452")]:
    sub = (r[r["balance"] == bal].sort_values(f"{MAIN_METRIC}_cv", ascending=False)
           .drop_duplicates("max_depth").set_index("max_depth").loc[order])
    ax.plot(order, sub[f"{MAIN_METRIC}_train"], "o--", color=color, label=f"eğitim ({bal})")
    ax.plot(order, sub[f"{MAIN_METRIC}_cv"], "o-", color=color, label=f"CV ({bal})")
ax.set_xlabel("max_depth")
ax.set_ylabel(MAIN_METRIC)
ax.set_title("Decision Tree: derinliğin etkisi")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / f"{MODEL_CODE}_depth_overfitting.png", dpi=200)
plt.show()

## 4. Feature engineering'in etkisi

Ortak pipeline 5 yeni özellik ekler (17 → 28 sütun). Ağaçlar gereksiz sütunları bölmede kullanmayarak kendileri eleyebilir; yine de türetilen özelliklerin katkı sağlayıp sağlamadığı en iyi parametrelerle kontrol edilir (bu deneyler `results.csv`'ye yazılmaz).

In [8]:
rows = {}
for fe in [True, False]:
    pipe = build_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE, **best_params),
                          balancing=chosen_balancing, feature_engineering=fe)
    res = cv_evaluate(pipe, X_train, y_train, model_name=f"{MODEL_NAME} (FE={fe})", save=False, verbose=False)
    rows["FE açık" if fe else "FE kapalı"] = res["summary"]["cv_mean"]
pd.DataFrame(rows).T.round(3)

metric,recall,precision,f1,specificity,roc_auc,pr_auc,accuracy
FE açık,0.546,0.147,0.230,0.692,0.662,0.168,0.680
FE kapalı,0.552,0.137,0.219,0.668,0.653,0.154,0.658


## 5. Final model: CV değerlendirmesi

Seçilen model ortak `cv_evaluate` ile değerlendirilir ve sonucu `results.csv`'ye **bir satır** olarak eklenir. Karşılaştırma notebook'u bu satırı kullanır. Eğitim ve CV skorları arasındaki fark (`gap`) overfitting göstergesidir.

In [9]:
cv_result = cv_evaluate(best, X_train, y_train, model_name=MODEL_NAME,
                        note=f"{chosen_balancing} + DT, GridSearch: {best_params}")

===== Decision Tree: 5 katlı CV (eşik = 0.5) =====
Kat başına ortalama eğitim süresi: 0.02 sn

             train_mean  cv_mean  cv_std    gap
metric                                         
recall            0.654    0.546   0.090  0.108
precision         0.175    0.147   0.021  0.028
f1                0.274    0.230   0.022  0.044
specificity       0.702    0.692   0.071  0.010
roc_auc           0.743    0.662   0.026  0.081
pr_auc            0.190    0.168   0.045  0.022
accuracy          0.698    0.680   0.058  0.019

Sonuç 'results.csv' dosyasına eklendi.


## 6. Eşik seçimi

Varsayılan eşik 0.5'tir, ama inmeyi kaçırmak yanlış alarmdan pahalı olduğu için eşik, **test setine bakılmadan** CV tahminleri (OOF) üzerinde seçilir: hedef recall'u (%80) en az yanlış alarmla sağlayan eşik.

Not: Decision Tree'nin skoru yaprağın inme oranıdır; ağacın yaprak sayısı kadar farklı skor olabilir. Bu yüzden eşik tablosunda bazı eşikler aynı sonucu verir.

In [10]:
table, threshold = threshold_analysis(cv_result["y"], cv_result["oof_score"], model_name=MODEL_NAME)

===== Decision Tree: Eşik analizi (toplam 306 inme, 3521 kişi) =====
 threshold  caught  missed  false_alarms  recall  precision  specificity  false_alarms_per_1000
      0.05     276      30          2570   0.902      0.097        0.201                729.906
      0.10     262      44          2197   0.856      0.107        0.317                623.970
      0.15     253      53          2046   0.827      0.110        0.364                581.085
      0.20     243      63          1887   0.794      0.114        0.413                535.927
      0.25     233      73          1733   0.761      0.119        0.461                492.190
      0.30     225      81          1554   0.735      0.126        0.517                441.352
      0.35     208      98          1371   0.680      0.132        0.574                389.378
      0.40     201     105          1303   0.657      0.134        0.595                370.065
      0.45     182     124          1113   0.595      0.141        

## 7. Overfitting kontrolü: öğrenme eğrisi

Eğitim verisi büyüdükçe eğitim ve CV skorları birbirine yaklaşıyorsa model genelleşiyor demektir. Büyük ve kapanmayan bir fark overfitting, ikisinin birlikte düşük kalması underfitting gösterir.

In [11]:
lc = learning_curve_analysis(best, X_train, y_train, model_name=MODEL_NAME)
lc.round(3)

,train_size,train_mean,cv_mean,cv_std
0,281,0.276,0.125,0.017
1,788,0.236,0.139,0.013
2,1295,0.245,0.134,0.006
3,1802,0.196,0.151,0.019
4,2309,0.196,0.146,0.009
5,2816,0.169,0.145,0.011


## 8. Test seti (yalnızca bir kez)

Final model tüm eğitim verisiyle yeniden eğitilir, seçilen eşikle test setinde değerlendirilir ve `models/` klasörüne kaydedilir.

In [12]:
best.fit(X_train, y_train)
test_result = test_evaluate(best, X_test, y_test, threshold=threshold,
                            model_name=MODEL_NAME, cv_result=cv_result)

joblib.dump(best, MODEL_DIR / f"{MODEL_CODE}.joblib")
print(f"Model kaydedildi: models/{MODEL_CODE}.joblib")

===== Decision Tree: Test seti (eşik = 0.179) =====
68/77 inme yakalandı, 9 inme kaçırıldı, 483 yanlış alarm (881 kişi).

             cv_same_threshold   test  cv_minus_test
recall                   0.810  0.883         -0.073
precision                0.112  0.123         -0.011
f1                       0.197  0.217         -0.019
specificity              0.390  0.399         -0.010
roc_auc                  0.660  0.698         -0.038
pr_auc                   0.156  0.158         -0.002
accuracy                 0.426  0.442         -0.015
Model kaydedildi: models/dt.joblib


## 9. Ağacın ilk seviyeleri

Final ağacın ilk 3 seviyesi. Her kutuda soru, o düğüme düşen eğitim verisinin yüzdesi (`samples`) ve sınıf oranları (`value` = [inme yok, inme var]) görünür. Renk koyulaştıkça düğüm daha saf olur (turuncu: inme yok, mavi: inme var). Dengeleme yüzünden kökte oranlar [0.5, 0.5] görünür: SMOTE'ta sentetik hastalar eklenmiştir, class_weight'te inme vakaları ağır sayılmıştır.

Sütunlar ölçeklenmiş olduğu için eşikler ham değer değil, standart sapma cinsindendir (ör. `AGE <= 0.8`: ortalamanın 0.8 standart sapma üstü).

In [13]:
tree = best.named_steps["model"]
print("Derinlik:", tree.get_depth(), "| yaprak sayısı:", tree.get_n_leaves())

fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(tree, max_depth=3, feature_names=list(tree.feature_names_in_), class_names=CLASS_NAMES,
          filled=True, impurity=False, proportion=True, fontsize=9, ax=ax)
plt.tight_layout()
plt.savefig(FIG_DIR / f"{MODEL_CODE}_tree.png", dpi=200)
plt.show()

Derinlik: 5 | yaprak sayısı: 26


## 10. XAI: modelin açıklanması

Decision Tree için ortak `explain()` fonksiyonu **TreeExplainer** kullanır: hızlıdır, tüm test verisi açıklanır. Fonksiyona aramanın kendisi değil, `best` (pipeline) verilir.

**Grafik okuma:** Bar grafiğinde çubuk uzunluğu önemi gösterir. Özet grafiğinde kırmızı = özelliğin değeri yüksek, mavi = düşük; nokta sağdaysa inme riskini artırıyor. LIME'da yeşil çubuklar "inme var" kararını destekleyen koşullardır.

In [14]:
# explain() shows its plots in order: SHAP bar, SHAP summary, LIME. Each one is also saved to figures/.
_names = iter([f"{MODEL_CODE}_shap_bar.png", f"{MODEL_CODE}_shap_summary.png", f"{MODEL_CODE}_lime.png"])
_show = plt.show
def _save_and_show(*args, **kwargs):
    name = next(_names, None)
    if name:
        plt.gcf().savefig(FIG_DIR / name, dpi=200, bbox_inches="tight")
    _show(*args, **kwargs)

plt.show = _save_and_show
try:
    xai = explain(best, X_train, X_test, model_name=MODEL_NAME)
finally:
    plt.show = _show

===== Decision Tree: XAI Analizi =====



En önemli 5 özellik (SHAP):
          feature  mean_abs_shap
              AGE       0.156213
comorbidity_count       0.076199
          CIGPDAY       0.057752
            DIABP       0.047464
         HEARTRTE       0.045192

[Decision Tree] Test hastası #2: İnme olasılığı = 74.55%


In [15]:
xai["shap_importance"].head(10)

,feature,mean_abs_shap
0,AGE,0.156213
1,comorbidity_count,0.076199
2,CIGPDAY,0.057752
3,DIABP,0.047464
4,HEARTRTE,0.045192
5,educ,0.039300
6,SEX_female,0.034103
7,CURSMOKE,0.027469
8,PREVCHD,0.014601
9,BPMEDS,0.011313


## 11. Özet (rapor için)

### Neden Decision Tree?

- **Yorumlanabilirlik:** Decision Tree kararını "yaş eşiğin üstünde mi, eşlik eden hastalık var mı?" gibi bir soru zinciriyle verir. Ağaç çizilerek modelin bir hasta için hangi yoldan karara vardığı doğrudan okunabilir. Gerekçesiz kararın tıpta işe yaramadığı bu projede, kendi başına açıklanabilen bir model olarak seçildi.
- **Ön işleme ihtiyacı az:** Ağaçlar ölçeklemeden etkilenmez, sayısal ve ikili (0/1) sütunları birlikte kullanabilir, özellikler arasındaki etkileşimleri (ör. "yaşlı ve tansiyonu yüksek") kendiliğinden yakalar.
- **Ağaç ailesinin temel taşı:** Random Forest çok sayıda Decision Tree'nin birleşiminden oluşur. Tek ağacın sonuçları, Random Forest'ın tek ağaca göre ne kazandırdığını ölçmek için bir referans çizgisi olarak kullanıldı.
- **Bilinen zayıflığı:** Sınırlanmayan bir ağaç eğitim verisini ezberler (overfitting) ve verideki küçük değişikliklere karşı kararsızdır. Bu yüzden ağacın büyümesi hiperparametrelerle sınırlandı.

### Yöntem

- Hiperparametreler (`max_depth`, `min_samples_leaf`, `criterion`) ve dengeleme yöntemi (SMOTE ya da `class_weight="balanced"`) birlikte, 192 kombinasyonluk bir GridSearch ile, ekibin ortak 5 katlı CV bölmesi üzerinde arandı. En iyi model PR-AUC ile seçildi.
- Seçilen model: **derinlik 5, yaprak başına en az 80 kişi, `entropy`, SMOTE** (26 yaprak). SMOTE ile en iyi `class_weight` ayarı arasındaki fark çok küçüktür (CV PR-AUC 0,168 ve 0,165; standart sapma 0,045), yani iki dengeleme yöntemi bu modelde benzer sonuç verir.
- Derinlik grafiği (`dt_depth_overfitting.png`): derinlik arttıkça eğitim skoru yükselmeye devam ederken CV skoru derinlik 5'ten sonra düşmektedir. Bu, daha derin ağaçların eğitim verisini ezberlemeye başladığını gösterir.
- Feature engineering ile türetilen özellikler CV PR-AUC'yi 0,154'ten 0,168'e çıkardı; bu yüzden ortak pipeline'daki haliyle kullanıldı.

### Sonuçlar

| | CV (5 kat, eşik 0,5) | CV (seçilen eşik 0,18) | Test (881 kişi, eşik 0,18) |
|---|---|---|---|
| Recall | 0,546 | 0,810 | 0,883 |
| Precision | 0,147 | 0,112 | 0,123 |
| Specificity | 0,692 | 0,390 | 0,399 |
| ROC-AUC | 0,662 | | 0,698 |
| PR-AUC | 0,168 | | 0,158 |

- Eşik, test setine bakılmadan CV tahminleri üzerinde, vakaların en az %80'ini en az yanlış alarmla yakalayacak şekilde 0,18 seçildi.
- Test setinde 77 inme vakasının **68'i yakalandı, 9'u kaçırıldı**; bunun bedeli 804 sağlıklı kişiden 483'üne verilen yanlış alarmdır.
- Eğitim ile CV skorları arasındaki fark küçüktür (PR-AUC 0,190 / 0,168) ve öğrenme eğrisinde iki skor veri arttıkça birbirine yaklaşmaktadır: model ezberlememiştir. CV ve test skorlarının birbirine yakın olması, modelin yeni veriye tutarlı genellendiğini gösterir.

### Model neye dayanıyor?

- Ağacın ilk bölünmesi **yaşa** göredir; yaşlı grupta ikinci soru **eşlik eden hastalık sayısı** (diyabet, hipertansiyon, koroner kalp hastalığı), genç grupta **diyastolik tansiyondur**.
- SHAP'a göre en önemli özellikler: yaş, eşlik eden hastalık sayısı, günlük sigara sayısı, diyastolik tansiyon ve nabız. Bunlar bilinen inme risk faktörleriyle uyumludur.

### Değerlendirme

- Decision Tree rastgele tahminden belirgin şekilde iyidir (PR-AUC 0,17; verideki inme oranına eşit olan rastgele tahmin düzeyi 0,09), ancak tek başına sınırlı bir modeldir: vakaların çoğunu yakalamak için çok sayıda yanlış alarm vermektedir (precision %12).
- Bunun nedeni, kararın 5 soruluk kaba bir kural zincirine indirgenmesi ve tek ağacın küçük veri değişikliklerine duyarlı olmasıdır. Random Forest, çok sayıda ağacın oylarını birleştirerek bu iki sorunu azaltmayı hedefler.